# Modül 04: Bilgisayarlı Görü — Giriş
### CNN ile Görüntü Sınıflandırma Temelleri

---

**Proje:** Eğitim Teknolojileri için PyTorch Eğitimi  
**Hazırlayan:** Uğur Sırvermez — Bursa Uludağ Üniversitesi  
**Lisans:** [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)  
**Repo:** [github.com/ugursirvermez/PyTorch_Education](https://github.com/ugursirvermez/PyTorch_Education)

> Bu modül, [Daniel Bourke](https://github.com/mrdbourke)'ün  
> **[Zero to Mastery Learn PyTorch for Deep Learning](https://github.com/mrdbourke/pytorch-deep-learning)**  
> ([learnpytorch.io](https://www.learnpytorch.io)) adlı çalışmasından ilham alınarak hazırlanmıştır.  
> İçerik Türkçeye çevrilmiş, eğitim teknolojistleri için yeniden yapılandırılmış ve pedagojik modül formatına dönüştürülmüştür.

---

## Ogrenme Hedefleri

- TorchVision kütüphanesini kullanarak hazır veri setlerini (FashionMNIST) yükleyebilirsiniz.
- DataLoader ile mini-batch eğitimi yapılandırabilirsiniz.
- Basit bir tam bağlantılı sinir ağı (dense network) kurarak görüntü sınıflandırabilirsiniz.
- Eğitim sürecini görselleştirebilir ve model performansını yorumlayabilirsiniz.

---

## Ön Koşullar

Modül 03 tamamlanmış olmalıdır. Görüntünün piksel matrisi olduğuna dair temel kavrayış yeterlidir.

---

⏱ Tahmini süre: 3–3.5 saat

---

## Modül Özeti

Bilgisayarlı görüye giriş modülüdür. TorchVision araçlarını ve DataLoader altyapısını tanıyarak görüntü verisiyle çalışmanın temelini atıyorsunuz. Sonraki modülde bu temelin üzerine CNN katmanları inşa edilecek.

---


<a href="https://colab.research.google.com/github/ugursirvermez/PyTorch_Education/blob/main/04_pytorch_computer_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Kütüphaneler
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor
import torch
import torchvision
import matplotlib.pyplot as plt
from helper_functions import plot_predictions, plot_decision_boundary, accuracy_fn #Daha önceden yazdığımız kodlardan alınma aslında.
from pathlib import Path
from timeit import default_timer as timer
from timeit import default_timer as timer # Zaman ayarları için var.
from tqdm.auto import tqdm #Eğitimi işlem barında göstermek için kullanacağız. Progess bar kütüphanesi
import requests


# Computer Vision

In [ ]:
#PyTorch Computer Vision -> TorchVision
#TorchVision -> resimleri yorumlama dönüştürme ve içerikleri yordamak için kullanılan kütüphanedir.
#KUTUPHANELER
#-------------------------------------------------
#Veri Setimizi Oluşturalım.
#MNIST DataBase -> Makine Öğreniminde kullanılan eğitilmiş ve test görsellerinin kullanılmasında etkili olan bir veri tabanıdır.
# 60,000 eğitilmiş, 10,000 test edilmiş görsel içinde barındırabilir. 28x28 piksel barındırır.
#FASHIONMNIST -> sıklıkla bu projede kullanacağız.

#root = veri nereye inecek?, train = veri setini eğitecek miyiz?, download = veriyi indirecek miyiz?
#transform = veriyi nasıl dönüştürmek istiyoruz?, target_transform = etiketleri ve hedefleri nasıl değiştireceğiz?
#TRAIN_DATA
train_data = datasets.FashionMNIST(root="data", train=True, download=True, transform = torchvision.transforms.ToTensor(), target_transform = None)

#TEST_DATA
test_data = datasets.FashionMNIST(root="data", train=False, download=True, transform = torchvision.transforms.ToTensor(), target_transform = None)
#Yukarıdaki değişkenlerimizden sonra Google Colab veya bilgisyarınımızda FashionMNIST'ten klasörler indirilir.
#Bunları görelim.
print(f"Train Data:{len(train_data)}, Test Data:{len(test_data)}")


In [ ]:
#Tensörlerimizin durumunu kontrol edebiliriz.
image, label = train_data[0] #resimler ve etiketler train_data'dan al.
class_names = train_data.classes # veriler nasıl sınıflanmış bakalım.
#print (image, label) #isteğe bağlı
print(class_names)

In [ ]:
#Resimlerin şekline ve etiketine bakalım.
print(f"Image shape: {image.shape} -> [color_channels, height, width]")
print(f"Image Label: {class_names[label]}")

# Rastgele Örnekleri Sergileme

In [ ]:
#Rastgele Örnekleri Sergileme
print(f"Image Shape: {image.shape}")
plt.imshow(image.squeeze(), cmap="gray") #Görselleri çiz fakat verileri sıkıştırmamız gerekiyor.
plt.title(class_names[label])

In [ ]:
#Torch Random seed ile yazdıralım
torch.manual_seed(42) #ilk resmi hep aynı gerisini random atacak.
fig = plt.figure(figsize=(9,9))
rows, cols = 4,4
for i in range(1, rows*cols+1):
	random_idx = torch.randint(0, len(train_data), size=[1]).item()
	img, label = train_data[random_idx]
	fig.add_subplot(rows, cols, i)
	plt.imshow(img.squeeze(), cmap="gray")
	plt.title(class_names[label])
	plt.axis(False)

# DataLoader Hazırlama

In [ ]:
#DataLoader Hazırlama
#DataLoader -> PyTorch nesnesinden Python nesnesine çevirmeyi sağlar. Mini Batches oluşturacağız. Batch_size = 32 olmalı.

#İşlenecek Batch maksimum boyutu
BATCH_SIZE = 32

#Veri setini yenilemeye çevirelim. Train_Data
train_dataloader = DataLoader(dataset = train_data, batch_size = BATCH_SIZE, shuffle = True)
#Test_Data
test_dataloader = DataLoader(dataset = test_data, batch_size = BATCH_SIZE, shuffle = False)
#Eğitilmiş verilerin içinde ne var?
train_features_batch , train_labels_batch = next(iter(train_dataloader))

torch.manual_seed(42)
random_idx = torch.randint(0, len(train_features_batch), size=[1]).item()
img, label = train_features_batch[random_idx], train_labels_batch[random_idx]
plt.imshow(img.squeeze(), cmap="gray")
plt.title(class_names[label])
plt.axis(False)

# 2 Doğrusal Katman Tasarlama

In [ ]:
# Model_0 Tasarlama
flatten_model = nn.Flatten() #tensörün boyutları ile devam eden ve Sequential yerine kullanılır.

# Tek bir örnek alalım
x = train_features_batch[0]

#Flatten örnek
output = flatten_model(x) #forward pass olacak

print(f"Flatten öncesi Shape: {x.shape}")
print(f"Flatten sonrası Shape: {output.shape}")

# Modeli ve Döngüleri Uygulama: Model_0

In [ ]:
if Path("helper_functions.py").is_file():
	print("helper_functions.py varsa indirme")
else:
	print("helper_function.py indir")
	request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
	with open("helper_functions.py", "wb") as f: #aynı dosyayı aç ve içeriğini yaz.
		f.write(request.content)

#---------------------------------------------------KÜTÜPHANELER------------------------------------------------------

class FashionMNISTModelV0(nn.Module):
	def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
		super().__init__()
		self.layer_stack = nn.Sequential(nn.Flatten(),
                                     nn.Linear(in_features = input_shape, out_features = hidden_units),
																		 nn.Linear(in_features = hidden_units, out_features = output_shape))

	def forward(self, x):
		return self.layer_stack(x)

#Zamana bağlı eğitimi yazdırmak için var.
def print_train_time(start: float, end: float, device: torch.device = None):
	total_time = end - start
	print(f"Eğitim Zamanı: {device}: {total_time} Saniye")
	return total_time

#Modeli başlatalım
torch.manual_seed(42)
#input_shape 28x28 pikselin çarpımıdır. Output_shape ise etiketlerin sayısıdır yani 9'dur.
model_0 = FashionMNISTModelV0(input_shape = 784, hidden_units = 10, output_shape = len(class_names))
#Modeli işlemciye yollayalım.
model_0.to("cpu")

#Loss, Optimizer ve Değerlendirme (Evalution) Değerlerini Oluşturma
loss_fn = nn.CrossEntropyLoss() #Bunu yukarıdaki tablodan biliyoruz.

optimizer = torch.optim.SGD(params= model_0.parameters(), lr=0.1)

#Şimdi Deneyleri Yapacağız. Modelin performansını(Loss, Optimizer v.s.) ve ne kadar hızlı çalıştığına bakacağız.
# 1. Devir daim için döngüleri oluşturacağız.
# 2. Döngüyü eğitilmi batch'ten alıp, eğitim adımlarını uygulatıp, eğitim kaybını batch başına hesaplatma
# 3. Döngüde batch test etme ve 2.adımdakileri test için uygulama
# 4. Sonucu yazdırma
# 5. Zamana bağlı olarak yazdırma
#Zamanlayıcı çalıştıralım.
start_time = timer()

epochs = 3

for epoch in tqdm(range(epochs)):
	print(f"Epoch: {epoch}\n-----")
	train_loss = 0

	#Döngü içinde döngü yapalım.
	for batch, (X, y) in enumerate(train_dataloader): #X -> Image, y -> Label
		#Eğitime başlıyoruz.
		model_0.train()
		y_pred = model_0(X) #Forward edilecek.

		#Loss'u her batch için yapacağız.
		loss = loss_fn(y_pred, y)
		train_loss += loss # eğitim kaybını kümeleyelim.
		#Batch için Optimizer
		optimizer.zero_grad()
		#Loss Backward
		loss.backward()
		#Optimizer step
		optimizer.step()

		if batch % 400 == 0:
			print(f"{batch * len(X)} / {len(train_dataloader.dataset)} örneklem var.")

	#Train_dataloader'daki kayba bakalım.
	train_loss /= len (train_dataloader)

	#Test edelim
	test_loss, test_acc = 0, 0
	model_0.eval()
	with torch.inference_mode():
		for X_test,y_test in test_dataloader:
			test_pred = model_0(X_test) #Forward edelim

			test_loss += loss_fn(test_pred, y_test) #loss'u kümeleyelim.

			test_acc += accuracy_fn(y_true = y_test, y_pred = test_pred.argmax(dim=1))

		test_loss /= len(test_dataloader)

		#Batch başına test_acc
		test_acc /= len(test_dataloader)

	print(f"Train Loss: {train_loss: .4f} | Test Loss: {test_loss: .4f}, Test Acc: {test_acc: .4f}")

#Zamanlayıcının sonu
end_time = timer()
#Geçen süre ve işlem cihazı
train_time = print_train_time(start = start_time, end = end_time, device = str(next(model_0.parameters())))

# Tahmin Yürütme ve Değerlendirme Değerleri

In [ ]:
torch.manual_seed(42)
def eval_model(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, accuracy_fn):
	loss, acc = 0,0
	model.eval()
	with torch.inference_mode():
		for X,y in tqdm(data_loader):
		#tahminler yürütme
			y_pred = model(X)

			#Batch başına loss ve acc hesaplama
			loss += loss_fn(y_pred, y)
			acc += accuracy_fn(y_true = y, y_pred = y_pred.argmax(dim=1))

		#Ortalama batch başına loss ve acc değerini yazalım.
		loss /= len(data_loader)
		acc /= len(data_loader)
	return {"model_name": model.__class__.__name__, "model_loss": loss.item(), "model_acc": acc}

model_0_results = eval_model(model = model_0, data_loader = test_dataloader, loss_fn = loss_fn, accuracy_fn = accuracy_fn)
print(model_0_results)

# Doğrusal Olmayan Model: Model_1

In [232]:
#Model_1 inşa edelim.
#Hem Doğrusal hem doğrusal olmayan katmanlar olacak.
#Cihazı tanıtalım.
device = "cuda" if torch.cuda.is_available() else "cpu"

class FashionMNISTModelV1(nn.Module):
	def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
		super().__init__()
		self.layer_stack = nn.Sequential(nn.Flatten(),
																		 nn.Linear(in_features = input_shape, out_features = hidden_units),
																		 nn.ReLU(), # doğrusal olmayan çıktı için ReLU gereklidir.
																		 nn.Linear(in_features = hidden_units, out_features = output_shape),
																		 nn.ReLU()) #bir tane daha ekliyoruz.

	def forward(self, x):
		return self.layer_stack(x)

#Modelin Dışı - seed edelim
torch.manual_seed(42)
model_1 = FashionMNISTModelV1(input_shape=784, hidden_units=10, output_shape = len(class_names)).to(device)

#Loss ve Optimizer yapalım
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(params= model_1.parameters(), lr=0.1)

#Eğitim Döngülerini Oluşturalım.
# eğitin döngüsü -> train_Step()
# test döngüsü -> test_step()
# Eğitim Fonksiyonu
def train_step(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module,
							 optimizer: torch.optim.Optimizer, accuracy_fn, device: torch.device = device):

	train_loss, train_acc = 0,0

	#Eğitime başlıyoruz.
	model.train()

	#Döngü içinde döngü yapalım.
	for batch, (X, y) in enumerate(data_loader): #X -> Image, y -> Label
		#Cihazları ayarlayalım.
		X, y = X.to(device), y.to(device)

		y_pred = model_0(X) #Forward edilecek.

		#Loss'u her batch için yapacağız.
		loss = loss_fn(y_pred, y)
		train_loss += loss # eğitim kaybını kümeleyelim.
		train_acc = accuracy_fn(y_true = y, y_pred = y_pred.argmax(dim=1))

		#Batch için Optimizer
		optimizer.zero_grad()
		#Loss Backward
		loss.backward()
		#Optimizer step
		optimizer.step()

	#Train_dataloader'daki kayba bakalım.
	train_loss /= len (data_loader)
	train_acc /= len(data_loader)
	print(f"Train Loss: {train_loss: .5f} | Train Acc: {train_acc: .2f}")

# Test Fonksiyonu
def test_step(model: torch.nn.Module, data_loader: torch.utils.data.DataLoader, loss_fn: torch.nn.Module,
							optimizer: torch.optim.Optimizer, accuracy_fn, device: torch.device = device):

	test_loss, test_acc = 0, 0
	model.eval()
	# inference modu çalıştıralım.
	with torch.inference_mode():
		for X_test, y_test in data_loader:

			X_test, y_test = X_test.to(device), y_test.to(device)

			test_pred = model(X_test)

			test_loss += loss_fn(test_pred, y_test)
			test_acc += accuracy_fn(y_true = y, y_pred = test_pred.argmax(dim=1))

		test_loss /= len(data_loader)
		test_acc /= len(data_loader)
		print(f"Test Loss: {test_loss: .5f} | Test Acc: {test_acc:.2f}%\n")

In [ ]:
torch.manual_seed(42)

train_time_start_on_gpu = timer()

epochs = 3

for epoch in tqdm(range(epochs)):
	print(f"Epoch: {epoch}\n-----")
	train_step(model= model_1, data_loader = train_dataloader, loss_fn=loss_fn, optimizer = optimizer, accuracy_fn= accuracy_fn, device=device)
	test_step(model= model_1, data_loader = train_dataloader, loss_fn=loss_fn, optimizer = optimizer, accuracy_fn= accuracy_fn, device=device)

train_time_end_on_gpu = timer()
total_train_time_model_1 = print_train_time(start=train_time_start_on_gpu, end=train_time_end_on_gpu, device=device)